In [80]:
import os
os.chdir("/content")

In [81]:
import time
inicio_notebook = time.time()

### Instalación de requerimientos e importación de bibliotecas

In [82]:
%%writefile requirements.txt
contextily
gdown
pandas
geopandas
matplotlib
contextily
folium
polars
fastexcel

Overwriting requirements.txt


In [83]:
%pip install -r requirements.txt
import os
import gdown
import pandas as pd
import geopandas as gpd

import matplotlib.pyplot as plt
import contextily as ctx
import folium
import re

import polars as pl
import gc

from IPython.display import display

import zipfile
import glob
import shutil


### Archivos 2014-2021

In [84]:
#@title Operaciones de carpeta
#Definición de carpeta de destino para CSV operación del árbol de directorios

# Definir la carpeta y el archivo de destino
output_dir = "./content/csv"

# Crear la carpeta si no existe
os.makedirs(output_dir, exist_ok=True)

# Se cambia a la carpeta de descaga de CSV
os.chdir(output_dir)

In [85]:
#@title Fn para descarga de archivos
def descargar_archivos_desde_drive2(dict_base, dict_bk):
    """
    Descarga archivos de Google Drive utilizando un diccionario base.
    Si la descarga de un archivo falla, intenta descargarlo desde un diccionario de backup.
    Evita la descarga si el archivo ya existe localmente.

    Args:
        dict_base (dict): Diccionario principal con {nombre_archivo: id_drive}.
        dict_bk (dict): Diccionario de respaldo con {nombre_archivo: id_drive}.
    """
    for nombre_archivo, file_id_base in dict_base.items():

        # 1. Verificar si el archivo ya existe localmente
        if os.path.exists(nombre_archivo):
            print(f"El archivo '{nombre_archivo}' ya existe localmente. Se omite la descarga.")
            continue

        # 2. Intentar la descarga con el ID del diccionario base
        url_base = f'https://drive.google.com/uc?id={file_id_base}'
        print(f"\n[Base] Intentando descargar '{nombre_archivo}'...")

        try:
            gdown.download(url_base, output=nombre_archivo, quiet=False)
            print(f"¡Éxito! '{nombre_archivo}' descargado desde el servidor base.")

        except Exception as e_base:
            print(f"Error al descargar desde el servidor base: {e_base}")

            # 3. Plan de contingencia: Intentar con el diccionario de backup
            if nombre_archivo in dict_bk:
                file_id_bk = dict_bk[nombre_archivo]
                url_bk = f'https://drive.google.com/uc?id={file_id_bk}'
                print(f"[Backup] Intentando descarga de respaldo para '{nombre_archivo}'...")

                try:
                    gdown.download(url_bk, output=nombre_archivo, quiet=False)
                    print(f"¡Éxito! '{nombre_archivo}' descargado desde el servidor de backup.")
                except Exception as e_bk:
                    print(f"Fallo definitivo: No se pudo descargar '{nombre_archivo}' de ninguna fuente. Error: {e_bk}")
            else:
                print(f"Fallo definitivo: '{nombre_archivo}' no se encuentra en el diccionario de backup.")

In [86]:
#@title Diccionario de fuentes
dict_datasets_base = {

    'lineas-de-subte.csv': '1sWI3jP6f9VDJ-IE1EI1lkc8rkxooP3b7',
    'estaciones-accesibles.csv': '1V6Cjhf2QU_gcig6HT5EvXhk0n2Egerqr',
    'registro-historico-del-precio-del-boleto.csv': '1yETNbct23DLqYoN7ti6hNV3RdKErwMYI',
    'registro-historico-del-precio-del-boleto.xlsx': '1mY28zAPaI79Pt-OLoNSAhIxnCtflpmKU',
    'viajes_anual.csv': '1PEAW6Vik2k-J7k9C6gxNtl_2df636Q41',
    'historico_2014.csv': '1CyWPBgfAYRBcYvQO7U7cRlAoQbGWoosP',
    'historico_2015.csv': '1g7LpNJFqNcqDmgaGc81MMgV3VfrD6Ah6',
    'historico_2016.csv': '1QqOb3oLoMs014d2YBYw4_e8ww4jo-JX1',
    'historico_2017.csv': '1G7noINplTWyt2g9xRrS7l0BKgFOW05hv',
    'historico_2018.csv': '11WgJxZsC4zUURSlCUBEQKXCQK5RLkRNZ',
    'historico_2019.csv': '1DVgvubSgYCTPQCfA4zj5eiH_ni136o9A',
    'historico_2020.csv': '1hlfAVsJS20InzvIkUXT4t5_nOgd2m1NW',
    'historico_2021.csv': '1hW4qHioTzrXlDnBfWextpMQpar0kmL03'
}

dict_datasets_bk = {
    'lineas-de-subte.csv': '19DoKYs2KUBPENiXyPu9mj2qpngxgUxPD',
    'estaciones-accesibles.csv': '1YJ1Oa4ALUEpZ4mF8AeIWzy564bgcyXvV',
    'registro-historico-del-precio-del-boleto.csv': '1EE2rCerAGd585DO6qYIAX4O7O_QT5y5P',
    'registro-historico-del-precio-del-boleto.xlsx': '1EjW-d3NLb2sXgkiUlOx4qWn4tUlJs0c6',
    'viajes_anual.csv': '1HmXi6DDkfONEEkc_SNzmPgViUibroklX',
    'historico_2014.csv': '1glREzvff9Xy7QiRcQFbPwmLa7QtKdC8M',
    'historico_2015.csv': '1ubs3j1V-PTRbfEbn_D2x5OxBU0H7QIqt',
    'historico_2016.csv': '1PYDajEAWBBXK8Cb6R62al2dxpGdbuqtX',
    'historico_2017.csv': '1LPBerWrFf-wroVkcF1mcYyd-7rvwUu99',
    'historico_2018.csv': '17CKtd2ee9Rq9L8koSVtoHDwL1RdOWEWO',
    'historico_2019.csv': '1V4rfpMFFjnSOz8RQO5sgGDWkXx19C8Q5',
    'historico_2020.csv': '1_uCE3d_VMcbQHr39kKmBqtVyBNuddrFT',
    'historico_2021.csv': '1ptlXJ6Ky3cw89gBLmRsiTkf4P5G2c7Z-'
}

In [87]:
#@title Descarga de archivos

descargar_archivos_desde_drive2(dict_datasets_base, dict_datasets_bk)

El archivo 'lineas-de-subte.csv' ya existe localmente. Se omite la descarga.
El archivo 'estaciones-accesibles.csv' ya existe localmente. Se omite la descarga.
El archivo 'registro-historico-del-precio-del-boleto.csv' ya existe localmente. Se omite la descarga.
El archivo 'registro-historico-del-precio-del-boleto.xlsx' ya existe localmente. Se omite la descarga.
El archivo 'viajes_anual.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2014.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2015.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2016.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2017.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2018.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2019.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2020.csv' ya existe localmente. Se omite la descarga.
El archivo 'historico_2021.cs

### Carga de datasets accesorios y configuración global

Los datasets accesorios (líneas, estaciones, precios, viajes) son chicos y se mantienen en memoria todo el tiempo. Se cargan una sola vez.

In [88]:
#@title Configuración global polars y nulos

null_values = ["", " ", "null", "NULL", "NaN", "NA", "N/A", "-"]

# Config de visualización de Polars (no mutila strings largos)
pl.Config.set_tbl_rows(100)
pl.Config.set_fmt_str_lengths(100)

print('Polars ha sido configurado')

polars.config.Config

In [89]:
#@title Carga datasets accesorios

lineas_subte_df = pl.read_csv('lineas-de-subte.csv', encoding='latin-1', null_values=null_values)
estaciones_accesibles_df = pl.read_csv('estaciones-accesibles.csv', encoding='latin-1', null_values=null_values)
registro_historico_del_precio_del_boleto_df = pl.read_csv('registro-historico-del-precio-del-boleto.csv', encoding='latin-1', null_values=null_values)
registro_historico_del_precio_del_boleto_excel_df = pl.read_excel('registro-historico-del-precio-del-boleto.xlsx')
viajes_anual_df = pl.read_csv('viajes_anual.csv', encoding='latin-1', null_values=null_values)

print("Accesorios cargados:")
for n in ["lineas_subte_df","estaciones_accesibles_df","registro_historico_del_precio_del_boleto_df","viajes_anual_df"]:
    print(f"  • {n}: {globals()[n].shape}")


Accesorios cargados:
  • lineas_subte_df: (82, 3)
  • estaciones_accesibles_df: (93, 6)
  • registro_historico_del_precio_del_boleto_df: (304, 4)
  • viajes_anual_df: (48, 3)


### Exploración de estaciones accesibles sin escaleras mecánicas ni ascensores

In [90]:
# Exploración de estaciones accesibles sin escaleras mecánicas ni ascensores

# Se filtra usando .filter() y expresiones pl.col
resultado_df = estaciones_accesibles_df.filter(
    (pl.col("escaleras_mecanicas") == 0) & (pl.col("ascensores") == 0)
)

# Lo mostrás directamente (acordate que no hace falta display si es lo último)
resultado_df

long,lat,linea,estacion,escaleras_mecanicas,ascensores
f64,f64,str,str,i64,i64


### Observaciones

Se observa que no hay estaciones listadas como estaciones accesibles que posean 0 escaleras mecánicas y 0 ascensores, lo cual es correcto.

## Pipeline de procesamiento por año (carga → curación → exportación → liberación)

El dataset de cada año se procesa de forma **aislada**: se carga, se cura, se diagnostica, se exporta a Parquet y se libera la memoria antes de pasar al siguiente. En ningún momento conviven dos años en RAM.

Cada etapa de curación es una función pura (recibe un `DataFrame`, devuelve un `DataFrame`). La función maestra `procesar_anio()` las orquesta e imprime los diagnósticos mientras el año está en memoria.

### Archivos 2014-2021

### Definiciones y funciones de etapa

In [91]:
#@title EDA - historicos 2014 a 2021

# carga en un dataframe las primeras 5 lineas de cada archivos historico de 2014 a 2021
historico_2014_df = pd.read_csv('historico_2014.csv', encoding='latin-1', nrows=5)
historico_2015_df = pd.read_csv('historico_2015.csv', encoding='latin-1', nrows=5)
historico_2016_df = pd.read_csv('historico_2016.csv', encoding='latin-1', nrows=5)
historico_2017_df = pd.read_csv('historico_2017.csv', encoding='latin-1', nrows=5)
historico_2018_df = pd.read_csv('historico_2018.csv', encoding='latin-1', nrows=5)
historico_2019_df = pd.read_csv('historico_2019.csv', encoding='latin-1', nrows=5)
historico_2020_df = pd.read_csv('historico_2020.csv', encoding='latin-1', nrows=5)
historico_2021_df = pd.read_csv('historico_2021.csv', encoding='latin-1', nrows=5)

In [92]:
historico_2014_df

,FECHA,DESDE,HASTA,LINEA,MOLINETE,ID_ESTACION,ESTACION,PAX_PAGO,PAX_PASES_PAGOS,PAX_FRANQ,PAX_TOTAL
0,2014-04-09,09:15:00,09:29:00,B,LINEA_B_FLORIDA_E_TURN02,20,FLORIDA,7,NaN,NaN,7
1,2014-04-09,09:15:00,09:29:00,B,LINEA_B_FLORIDA_E_TURN03,20,FLORIDA,9,NaN,NaN,9
2,2014-04-09,09:15:00,09:29:00,B,LINEA_B_FLORIDA_O_TURN01,20,FLORIDA,14,NaN,2.0,16
3,2014-04-09,09:15:00,09:29:00,B,LINEA_B_FLORIDA_O_TURN02,20,FLORIDA,18,NaN,NaN,18
4,2014-04-09,09:15:00,09:29:00,B,LINEA_B_FLORIDA_O_TURN03,20,FLORIDA,12,NaN,NaN,12


In [93]:
historico_2015_df

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201501,2015-01-01,05:00:00,05:15:00,LINEA_H,LINEA_H_CASEROS_NORTE_TURN01,CASEROS,0.0,0.0,0.0,0.0
1,201501,2015-01-01,05:30:00,05:45:00,LINEA_A,LINEA_A_MISERERE_S_TURN03,PLAZA MISERERE,0.0,0.0,0.0,0.0
2,201501,2015-01-01,05:30:00,05:45:00,LINEA_D,LINEA_D_CATEDRAL_E_ASC01,CATEDRAL,0.0,0.0,0.0,0.0
3,201501,2015-01-01,05:30:00,05:45:00,LINEA_D,LINEA_D_CONGRESOTUC_O_TURN01,CONGRESO DE TUCUMAN,0.0,0.0,0.0,0.0
4,201501,2015-01-01,06:00:00,06:15:00,LINEA_C,LINEA_C_INDEPEN_TURN02,INDEPENDENCIA,0.0,0.0,0.0,0.0


In [94]:
historico_2016_df

,FECHA,DESDE,HASTA,LINEA,MOLINETE,ID_ESTACION,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FREQ,TOTAL
0,02/01/2016,05:00:00,05:15:00,D,LINEA_D_9JULIO_S_TURN02,6,9 DE JULIO,1,0,0,1
1,02/01/2016,05:00:00,05:15:00,D,LINEA_D_9JULIO_S_TURN01,6,9 DE JULIO,2,0,0,2
2,05/01/2016,05:00:00,05:15:00,D,LINEA_D_9JULIO_N_TURN01,6,9 DE JULIO,1,0,0,1
3,06/01/2016,05:00:00,05:15:00,D,LINEA_D_9JULIO_S_TURN03,6,9 DE JULIO,2,0,0,2
4,06/01/2016,05:00:00,05:15:00,D,LINEA_D_9JULIO_S_TURN02,6,9 DE JULIO,1,0,0,1


In [95]:
historico_2017_df

,V1,FECHA,DESDE,HASTA,LINEA,MOLINETE,ID_ESTACION,ESTACION,PAX_PAGOS,PAX_PASES_PAGOS,PAX_FREQ,TOTAL
0,1,01/01/2017,08:00:00,08:15:00,D,LINEA_D_9JULIO_S_TURN02,6,9 DE JULIO,1,0,0,1
1,2,01/01/2017,08:00:00,08:15:00,D,LINEA_D_9JULIO_N_TURN02,6,9 DE JULIO,1,0,0,1
2,3,01/01/2017,08:00:00,08:15:00,D,LINEA_D_9JULIO_S_TURN01,6,9 DE JULIO,1,0,0,1
3,4,01/01/2017,08:15:00,08:30:00,D,LINEA_D_9JULIO_S_TURN02,6,9 DE JULIO,1,0,0,1
4,5,01/01/2017,08:15:00,08:30:00,D,LINEA_D_9JULIO_S_TURN01,6,9 DE JULIO,2,0,0,2


In [96]:
historico_2018_df

,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total,periodo
0,2018-01-01,08:00:00,08:15:00,LineaA,LineaA_CBarros_S_Turn01,Castro Barros,1.0,0.0,0.0,1.0,201801
1,2018-01-01,08:00:00,08:15:00,LineaA,LineaA_Lima_S_Turn03,Lima,4.0,0.0,0.0,4.0,201801
2,2018-01-01,08:00:00,08:15:00,LineaA,LineaA_Pasco_Turn01,Pasco,1.0,0.0,0.0,1.0,201801
3,2018-01-01,08:00:00,08:15:00,LineaA,LineaA_Peru_S_Turn01,Peru,4.0,0.0,0.0,4.0,201801
4,2018-01-01,08:00:00,08:15:00,LineaA,LineaA_PJunta_S_Turn02,Primera Junta,2.0,0.0,0.0,2.0,201801


In [97]:
historico_2019_df

,periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total
0,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Lima_N_Turn02,Lima,1.0,0.0,0.0,1.0
1,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Loria_N_Turn03,Loria,3.0,0.0,0.0,3.0
2,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_Q_HALL_Turn01,Plaza Miserere,3.0,0.0,0.0,3.0
3,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_S_Turn01,Plaza Miserere,6.0,0.0,0.0,6.0
4,201901,2019-01-01,08:00:00,08:15:00,LineaA,LineaA_Miserere_S_Turn03,Plaza Miserere,10.0,0.0,0.0,10.0


In [98]:
historico_2020_df

,FECHA,DESDE,HASTA,LINEA,MOLINETE,ESTACION,pax_pagos,pax_pases_pagos,pax_franq,pax_TOTAL
0,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Acoyte_N_Turn01,Acoyte,1,0,0,1
1,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Carabobo_E_Turn02,Carabobo,6,0,0,6
2,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_CBarros_N_Turn03,Castro Barros,3,0,1,4
3,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_CBarros_S_Turn02,Castro Barros,2,0,0,2
4,01/01/2020,08:00:00,08:15:00,LineaA,LineaA_Congreso_N_Turn03,Congreso,2,0,0,2


In [99]:
historico_2021_df

,periodo;FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL
0,2021;1/1/2021;08:00:00;08:15:00;LineaA;LineaA_...
1,2021;1/1/2021;08:00:00;08:15:00;LineaA;LineaA_...
2,2021;1/1/2021;08:00:00;08:15:00;LineaB;LineaB_...
3,2021;1/1/2021;08:00:00;08:15:00;LineaB;LineaB_...
4,2021;1/1/2021;08:00:00;08:15:00;LineaB;LineaB_...


Tras esta exploración inicial se puede observar que los archivos historicos 2014 a 2021 si bien siguen una estructura general homóloga no respetan el mismo orden de columnas ni los nombres de las mismas se encuentran normalizados e incluso utilizan diferentes tipos de separadores (, o ;) por lo que es necesario aplicar transformaciones para resolverlo.

In [100]:
# se eliminan los dataframes cargados para exploración inicial
import gc

if "historico_2014_df" in locals() and historico_2014_df is not None:
    del historico_2014_df

if "historico_2015_df" in locals() and historico_2015_df is not None:
    del historico_2015_df

if "historico_2016_df" in locals() and historico_2016_df is not None:
    del historico_2016_df

if "historico_2017_df" in locals() and historico_2017_df is not None:
    del historico_2017_df

if "historico_2018_df" in locals() and historico_2018_df is not None:
    del historico_2018_df

if "historico_2019_df" in locals() and historico_2019_df is not None:
    del historico_2019_df

if "historico_2020_df" in locals() and historico_2020_df is not None:
    del historico_2020_df

if "historico_2021_df" in locals() and historico_2021_df is not None:
    del historico_2021_df

gc.collect()
print("Dataframes de exploracion inicial borrados - Memoria liberada")

Dataframes de exploracion inicial borrados - Memoria liberada


In [101]:
# Constantes del pipeline

#COLUMNAS_DESEADAS = [
#    "FECHA", "DESDE", "HASTA", "LINEA", "ESTACION", "PAX_PAGO",
#    "PAX_PAGOS", "PAX_PASES_PAGOS",
#    "PAX_FREQ", "PAX_FRANQ", "PAX_TOTAL", "TOTAL"
#]

COLUMNAS_DESEADAS = [
    "FECHA", "DESDE", "HASTA", "LINEA", "ESTACION", "PAX_TOTAL", "TOTAL"
]


# Carpeta de salida de los parquet (un nivel arriba del cwd ./content/csv -> ./content/parquet)
CARPETA_PARQUET = os.path.abspath(os.path.join(os.getcwd(), "..", "parquet"))
os.makedirs(CARPETA_PARQUET, exist_ok=True)
print(f"Los parquet se exportarán a: {CARPETA_PARQUET}")


Los parquet se exportarán a: /content/content/parquet


## Etapas del pipeline detransformacion

In [102]:
#@title ETAPA 1: Carga selectiva de columnas ---

def cargar_anio(anio: int) -> pl.DataFrame | None:
    """Carga un CSV anual trayendo solo las columnas de interés.
    Maneja los casos especiales: 2021 (separador ';') y 2016/2017 (encoding lossy)."""
    ruta_csv = f"historico_{anio}.csv"
    separador = ";" if anio == 2021 else ","
    encoding = "utf8-lossy" if anio in (2016, 2017) else "utf8"

    try:
        lf = pl.scan_csv(
            ruta_csv,
            null_values=null_values,
            infer_schema_length=0,   # todo como String; casteamos nosotros después
            separator=separador,
            encoding=encoding,
        )
        columnas_reales = lf.collect_schema().names()
        columnas_a_importar = [c for c in columnas_reales if c.upper() in COLUMNAS_DESEADAS]
        df = lf.select(columnas_a_importar).collect()
        print(f"  [carga] {ruta_csv} | sep='{separador}' enc='{encoding}' | "
              f"cols={columnas_a_importar} | shape={df.shape}")
        return df
    except FileNotFoundError:
        print(f"  [carga] No se encontró {ruta_csv} — se omite el año {anio}.")
        return None


In [103]:
#@title ETAPA 2: Estandarización de nomenclaturas y mayúsculas ---

def estandarizar_columnas(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Unifica nombres: PAX_FREQ/PAX_FRANQUICIAS -> PAX_FRANQ, TOTAL -> PAX_TOTAL,
    PAX_PAGO -> PAX_PAGOS (caso 2014), y finalmente todo a MAYÚSCULAS."""
    renombrado = {}
    cols_upper = [c.upper() for c in df.columns]

    for col in df.columns:
        cu = col.upper()
 #      if cu in ("PAX_FREQ", "PAX_FRANQUICIAS"):
 #         renombrado[col] = "PAX_FRANQ"
        # TOTAL -> PAX_TOTAL solo si no existe ya PAX_TOTAL (corrige el bug de indentación original)
 #      elif cu == "TOTAL" and "PAX_TOTAL" not in cols_upper:
        if cu == "TOTAL" and "PAX_TOTAL" not in cols_upper:
            renombrado[col] = "PAX_TOTAL"
#       elif cu == "PAX_PAGO":   # caso 2014: singular -> plural
#           renombrado[col] = "PAX_PAGOS"

    if renombrado:
        df = df.rename(renombrado)
        print(f"  [nomenclatura] {anio}: {renombrado}")

    # Todo a mayúsculas (idempotente)
    df = df.rename(lambda c: c.upper())
    return df


In [104]:
#@title ETAPA 3: Tipado de fechas + imputación de PAX con 0 ---

def tipar_fechas_y_pax(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Castea columnas PAX a Float64 (imputando nulos con 0) y FECHA a pl.Date
    con parseo elástico (ISO / latino / yanki)."""
    col_fecha = "FECHA"
    columnas_pax = [c for c in df.columns if "PAX_" in c or c == "TOTAL"]

    # Filtramos filas con FECHA vacía/nula
    df = df.filter(
        (pl.col(col_fecha).str.strip_chars() != "") & (pl.col(col_fecha).is_not_null())
    )

    # PAX -> float, limpiando espacios, e imputando 0
    operaciones = []
    for c in columnas_pax:
        if df.schema[c] == pl.String:
            expr = pl.col(c).str.replace_all(" ", "").cast(pl.Float64, strict=False)
        else:
            expr = pl.col(c)
        operaciones.append(expr.fill_null(0))
    if operaciones:
        df = df.with_columns(operaciones)

    # FECHA -> Date con coalesce de formatos
    if df.schema[col_fecha] == pl.String:
        df = df.with_columns(
            pl.coalesce([
                pl.col(col_fecha).str.to_date(format="%Y-%m-%d", strict=False),
                pl.col(col_fecha).str.to_date(format="%d/%m/%Y", strict=False),
                pl.col(col_fecha).str.to_date(format="%m/%d/%Y", strict=False),
            ]).alias(col_fecha)
        )
    nulos_fecha = df[col_fecha].is_null().sum()
    print(f"  [tipado fecha/pax] {anio}: pax imputadas={columnas_pax} | nulos FECHA restantes={nulos_fecha}")
    return df


In [105]:
#@title ETAPA 4: Tipado de horas (DESDE/HASTA -> pl.Time) ---

def tipar_horas(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Convierte DESDE y HASTA a pl.Time, tolerando formato 'HH:MM' (le agrega ':00')."""
    cols_hora = [c for c in df.columns if c in ("DESDE", "HASTA")]
    transformaciones = {}
    for c in cols_hora:
        if df.schema[c] == pl.String:
            expr = pl.col(c).str.strip_chars()
            expr = pl.when(expr.str.len_chars() == 5).then(expr + ":00").otherwise(expr)
            transformaciones[c] = expr.str.to_time(format="%H:%M:%S", strict=False)
    if transformaciones:
        df = df.with_columns(**transformaciones)
        print(f"  [tipado hora] {anio}: convertidas {list(transformaciones.keys())} a pl.Time")
    return df


In [106]:
#@title ETAPA 5: Diagnóstico de nulos (solo reporta) ---

def diagnosticar_nulos(df: pl.DataFrame, anio: int) -> None:
    reporte = df.select([
        pl.all().null_count().name.suffix("_nulos"),
        (pl.col(pl.String).str.strip_chars() == "").sum().name.suffix("_vacios"),
    ]).transpose(include_header=True, header_name="Columna_Métrica", column_names=["Cantidad"])
    print(f"  [nulos] HISTÓRICO {anio}:")
    print(reporte)


In [107]:
#@title ETAPA 6: Correcciones específicas por año ---

def correccion_2018_bonifacio(df: pl.DataFrame) -> pl.DataFrame:
    """2018: la estación 'Taller Bonifacio' aparece con LINEA nula y PAX_TOTAL=0.
    Eliminamos esos registros espurios."""
    bonifacio = df.filter(pl.col("ESTACION") == "Taller Bonifacio")
    if bonifacio.height:
        print(f"  [corr 2018] Taller Bonifacio: {bonifacio.height} registros hallados, "
              f"se eliminan los de PAX_TOTAL=0")
    return df.filter(
        ~((pl.col("ESTACION") == "Taller Bonifacio") & (pl.col("PAX_TOTAL") == 0.0))
    )

def consolidar_duplicados_clave(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """2020/2021: filas idénticas y duplicados por clave espacio-temporal.
    Se eliminan idénticas y se suman las métricas por clave."""
    antes = df.height
    df = df.unique()
    df = (
#        df.group_by(["FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION"])
         df.group_by(["FECHA", "DESDE", "HASTA", "LINEA", "ESTACION"])
          .agg([
              #pl.col("PAX_PAGOS").sum(),
              #pl.col("PAX_PASES_PAGOS").sum(),
              #pl.col("PAX_FRANQ").sum(),
              pl.col("PAX_TOTAL").sum(),
          ])
    )
    print(f"  [consolidación] {anio}: {antes:,} -> {df.height:,} filas "
          f"(reducción {antes - df.height:,})")
    return df

def aplicar_correcciones_especificas(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    if anio == 2018:
        df = correccion_2018_bonifacio(df)
    if anio in (2020, 2021):
        df = consolidar_duplicados_clave(df, anio)
    return df


In [108]:
#@title ETAPA 7: Diagnóstico de duplicados (solo reporta) ---

def diagnosticar_duplicados(df: pl.DataFrame, anio: int) -> None:
    total = df.height
    dup_exactos = total - df.unique().height
    cols_clave = [c for c in df.columns
                # if c in ("FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION")]
                  if c in ("FECHA", "DESDE", "HASTA", "LINEA", "ESTACION")]
    dup_clave = total - df.unique(subset=cols_clave).height
    print(f"  [duplicados] {anio}: total={total:,} | idénticos={dup_exactos:,} | "
          f"por clave={dup_clave:,}")


In [109]:
#@title ETAPA 8: Diagnóstico de outliers (solo reporta) ---

def diagnosticar_outliers(df: pl.DataFrame, anio: int) -> None:
    # Numéricos
    col_total = next((c for c in df.columns if "PAX_TOTAL" in c), None)
    if col_total:
        s = df.select([
            pl.col(col_total).min().alias("min"),
            pl.col(col_total).max().alias("max"),
            pl.col(col_total).mean().alias("media"),
            pl.col(col_total).std().alias("std"),
            (pl.col(col_total) < 0).sum().alias("negativos"),
        ])
        print(f"  [outliers num] {anio} ({col_total}): "
              f"min={s['min'][0]} max={s['max'][0]:,} media={s['media'][0]:.2f} "
              f"std={s['std'][0]:.2f} negativos={s['negativos'][0]}")
        if s['max'][0] is not None and s['max'][0] > 15000:
            print(f"      ALERTA: máximo físicamente improbable para 15 min.")
        if s['negativos'][0] > 0:
            print(f"      ALERTA: valores negativos presentes.")

    # Temporales
    dft = df.filter(pl.col("FECHA").is_not_null())
    fuera_anio = dft.filter(pl.col("FECHA").dt.year() != anio).height
    horas_rotas = dft.filter(pl.col("DESDE").is_null()).height
    print(f"  [outliers temp] {anio}: fechas fuera de año={fuera_anio} | horas DESDE nulas={horas_rotas}")
    if fuera_anio > 0:
        anios_det = dft.filter(pl.col("FECHA").dt.year() != anio)\
                       .select(pl.col("FECHA").dt.year()).unique().to_series().to_list()
        print(f"      ALERTA: años mezclados detectados: {anios_det}")


In [110]:
#@title ETAPA 9: Estandarización de LINEA ---

def estandarizar_linea(df: pl.DataFrame, anio: int) -> pl.DataFrame:
    """Extrae la letra de ramal (A-H) del final del string: 'LineaA'->'A', 'LINEA_A'->'A'."""
    df = df.with_columns(pl.col("LINEA").str.extract(r"([A-H])$", 1))
    unicos = df.select(pl.col("LINEA")).unique().sort("LINEA")["LINEA"].to_list()
    print(f"  [linea] {anio}: ramales -> {unicos}")
    return df


In [111]:
#@title FUNCIÓN MAESTRA: procesa un año de punta a punta y libera la memoria ---

def procesar_anio(anio: int, verbose_nulos: bool = False) -> bool:
    """Carga, cura, diagnostica y exporta a Parquet un único año.
    Devuelve True si se procesó, False si el archivo no existía.
    Al terminar, el DataFrame queda fuera de scope y se libera con gc.collect()."""
    print(f"\n{'='*70}\nPROCESANDO AÑO {anio}\n{'='*70}")

    df = cargar_anio(anio)
    if df is None:
        return False

    # Curación
    df = estandarizar_columnas(df, anio)
    df = tipar_fechas_y_pax(df, anio)
    df = tipar_horas(df, anio)

    # Diagnóstico de nulos (el detalle de la tabla solo si se pide)
    if verbose_nulos:
        diagnosticar_nulos(df, anio)

    # Correcciones específicas (2018 Bonifacio, 2020/2021 consolidación)
    df = aplicar_correcciones_especificas(df, anio)

    # Diagnósticos finales
    diagnosticar_duplicados(df, anio)
    diagnosticar_outliers(df, anio)

    # Estandarización de LINEA (después de eliminar nulos de Bonifacio)
    df = estandarizar_linea(df, anio)

    # Exportación
    ruta_parquet = os.path.join(CARPETA_PARQUET, f"historico_{anio}.parquet")
    df.write_parquet(ruta_parquet, compression="snappy")
    print(f"  [export] {ruta_parquet} ({df.height:,} filas)")

    # Liberación explícita
    del df
    gc.collect()
    print(f"  [memoria] año {anio} liberado.")
    return True


## Carga de dataset desde .parquet

In [112]:
# --- EJECUCIÓN INTELIGENTE: un año por vez con bypass por caché ---

# 1. Chequeamos si ya existen los 8 parquets consolidados en el disco
archivos_parquet_esperados = [f"historico_{anio}.parquet" for anio in range(2014, 2026)]
archivos_en_carpeta = os.listdir(CARPETA_PARQUET) if os.path.exists(CARPETA_PARQUET) else []

# Verificamos si todos los archivos de la lista están presentes en la carpeta
ya_existen_todos = all(p in archivos_en_carpeta for p in archivos_parquet_esperados)

print("======================================================================")
print("CHEQUEO DE CACHÉ DE PROCESAMIENTO (ETAPA DE SEGURIDAD)")
print("======================================================================")

if ya_existen_todos:
    print("CACHE DETECTADO: Todos los históricos ya están convertidos a Parquet.")
    print(f"Se saltea la lectura de CSVs crudos. Continuá directo a la conexión Lazy.")
    procesados = range(2014, 2026) # Simplemente para el print final
else:
    print("Caché incompleto o inexistente. Iniciando pipeline de curación por año...")

    procesados = []
    for anio in range(2014, 2026):
        # Ejecuta solo si el archivo parquet específico de ese año no existe
        # (por si se te cortó el proceso a la mitad en una corrida anterior)
        ruta_especifica = os.path.join(CARPETA_PARQUET, f"historico_{anio}.parquet")

        if os.path.exists(ruta_especifica):
            print(f"Histórico {anio}.parquet ya existe en disco. Omitiendo conversión.")
            procesados.append(anio)
            continue

        if procesar_anio(anio, verbose_nulos=False):
            procesados.append(anio)
        gc.collect()  # Seguro adicional entre años

print(f"\n{'='*70}")
print(f"Pipeline finalizado. Estado de Parquets listos en disco: {list(procesados)}")
print(f"{'='*70}\n")

CHEQUEO DE CACHÉ DE PROCESAMIENTO (ETAPA DE SEGURIDAD)
CACHE DETECTADO: Todos los históricos ya están convertidos a Parquet.
Se saltea la lectura de CSVs crudos. Continuá directo a la conexión Lazy.

Pipeline finalizado. Estado de Parquets listos en disco: [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]



### Archivos 2022-2025

In [113]:
# 1. Se obtiene la ruta absoluta del directorio de trabajo (.../content/csv)
ruta_actual = os.getcwd()
ruta_actual

# 2. Se sube un nivel para posicionar el directorio de trabajo en /content de forma absoluta
ruta_content = os.path.abspath(os.path.join(ruta_actual, ".."))
ruta_content

# 3. Se define la ruta de la nueva carpeta 'zip' dentro de content
carpeta_zip = os.path.join(ruta_content, "zip")
carpeta_zip

# 4. Se crea el directorio de forma segura
os.makedirs(carpeta_zip, exist_ok=True)

os.chdir(carpeta_zip)

In [114]:
#@title Descarga de datasets 2022-2025

import os
import gdown
import requests


def descarga_22_25(nombre_archivo, id_drive, url_cdn):
    # 1. Verificar si el archivo ya existe
    if os.path.exists(nombre_archivo):
        print(f"✅ {nombre_archivo} ya existe. Omitiendo descarga.")
        return

    print(f"⏳ Procesando {nombre_archivo}...")

    # 2. Intentar descargar desde Google Drive
    try:
        print(f"👉 Intentando descargar desde Google Drive...")
        # gdown descarga directamente usando el ID
        url_drive = f"https://drive.google.com/uc?id={id_drive}"
        output = gdown.download(url_drive, nombre_archivo, quiet=False)

        if output:
            print(f"🎉 Descarga exitosa desde Google Drive: {nombre_archivo}")
            return
    except Exception as e:
        print(f"⚠️ Error al descargar desde Drive: {e}")

    # 3. Fallback: Intentar descargar desde el CDN si Drive falla
    try:
        print(
            f"🔄 Drive falló. Intentando descargar desde el CDN ({url_cdn})..."
        )
        response = requests.get(url_cdn, stream=True)
        response.raise_for_status()  # Lanza un error si la descarga falla (ej. error 404 o 500)

        with open(nombre_archivo, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"🎉 Descarga exitosa desde el CDN: {nombre_archivo}")

    except Exception as e:
        print(
            f"❌ Error crítico: No se pudo descargar {nombre_archivo} de ninguna fuente. Motivo: {e}"
        )


# --- Ejecución de las descargas ---

archivos_a_descargar = [
    {
        "nombre": "molinetes_2022.zip",
        "id_drive": "1GRoBZAvo6iS_HC1M2oqS8LCEmg8lKcZG",
        "url": "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2022.zip",
    },
    {
        "nombre": "molinetes_2023.zip",
        "id_drive": "1xW7Mnzo1zLapLe70AiH0zekDQS7MnT5N",
        "url": "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2023.zip",
    },
    {
        "nombre": "molinetes_2024.zip",
        "id_drive": "19dExiRLnwZrrt8shkWrZYZeokr2S8ksa",
        "url": "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2024.zip",
    },
    {
        "nombre": "molinetes_2025.zip",
        "id_drive": "1LdGNW6jJTpxghyLbiOgnz1eS2qnluGgN",
        "url": "https://cdn.buenosaires.gob.ar/datosabiertos/datasets/sbase/subte-viajes-molinetes/molinetes-2025.zip",
    },
]

for item in archivos_a_descargar:
    descarga_22_25(item["nombre"], item["id_drive"], item["url"])
    print("-" * 50)

✅ molinetes_2022.zip ya existe. Omitiendo descarga.
--------------------------------------------------
✅ molinetes_2023.zip ya existe. Omitiendo descarga.
--------------------------------------------------
✅ molinetes_2024.zip ya existe. Omitiendo descarga.
--------------------------------------------------
✅ molinetes_2025.zip ya existe. Omitiendo descarga.
--------------------------------------------------


In [115]:
print("==================================================================")
print("EXTRACCIÓN HISTÓRICA: GRANULARIDAD POR PERIODO")
print("==================================================================")

# Se listan los zip descargados en la carpeta actual (content/zip)
archivos_zip = [f for f in os.listdir(".") if f.endswith(".zip")]

for nombre_zip in sorted(archivos_zip):
    # Se extrae el año del nombre (ej: 'molinetes_2022.zip' -> '2022')
    anio = nombre_zip.split("_")[1].split(".")[0]

    # Se define la ruta absoluta apuntando a /content/csv/molinetes_2022/
    # Se sube un nivel (..) para salir de 'zip' y se ingresa a 'csv/molinetes_ANIO'
    carpeta_destino = os.path.abspath(os.path.join("..", "csv", f"molinetes_{anio}"))
    os.makedirs(carpeta_destino, exist_ok=True)

    print(f"Descomprimiendo {nombre_zip}...")

    with zipfile.ZipFile(nombre_zip, 'r') as zip_ref:
        for miembro in zip_ref.infolist():
            # Se ignoran directorios internos del ZIP
            if miembro.is_dir():
                continue

            # Se filtran solo los archivos de datos CSV
            if miembro.filename.lower().endswith(".csv"):
                # Se aplana la estructura interna del ZIP
                nombre_archivo_plano = os.path.basename(miembro.filename)
                ruta_final_archivo = os.path.join(carpeta_destino, nombre_archivo_plano)

                # Se extrae de forma directa el flujo binario
                with zip_ref.open(miembro) as fuente, open(ruta_final_archivo, "wb") as destino:
                    shutil.copyfileobj(fuente, destino)

                print(f"   • Extraído: csv/molinetes_{anio}/{nombre_archivo_plano}")

print("\n==================================================================")
print("Estructura consolidada en /content/csv/ exitosamente.")
print("==================================================================")

EXTRACCIÓN HISTÓRICA: GRANULARIDAD POR PERIODO
Descomprimiendo molinetes_2022.zip...
   • Extraído: csv/molinetes_2022/202201_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202201_PAX15min-DEH.csv
   • Extraído: csv/molinetes_2022/202202_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202202_PAX15min-DEH.csv
   • Extraído: csv/molinetes_2022/202203_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202203_PAX15min-DEH.csv
   • Extraído: csv/molinetes_2022/202204_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202204_PAX15min-DEH.csv
   • Extraído: csv/molinetes_2022/202205_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202205_PAX15min-DEH.csv
   • Extraído: csv/molinetes_2022/202206_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202206_PAX15min-DEH.csv
   • Extraído: csv/molinetes_2022/202207_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202207_PAX15min-DEH.csv
   • Extraído: csv/molinetes_2022/202208_PAX15min-ABC.csv
   • Extraído: csv/molinetes_2022/202208_PAX1

In [116]:
# 1. Se define la ruta de la carpeta que queremos inspeccionar
carpeta_2022 = os.path.abspath(os.path.join("..", "csv", "molinetes_2022"))

print("==================================================================")
print("INSPECCIÓN DE ESQUEMA DE COLUMNAS - MOLINETES 2022")
print("==================================================================")
print(f"• Analizando archivos en: {carpeta_2022}\n")

if os.path.exists(carpeta_2022):
    # Se listan todos los CSVs internos
    archivos_csv = [f for f in os.listdir(carpeta_2022) if f.endswith(".csv")]

    if not archivos_csv:
        print("No se encontraron archivos CSV dentro de la carpeta.")

    for nombre_csv in sorted(archivos_csv):
        ruta_completa = os.path.join(carpeta_2022, nombre_csv)

        try:
            # Se escanean los metadatos de forma Lazy para no tocar la RAM
            lf_inspeccion = pl.scan_csv(ruta_completa, infer_schema_length=5)
            esquema = lf_inspeccion.collect_schema()

            print(f"Archivo: {nombre_csv}")
            print(f"   • Columnas detectadas: {esquema.names()}")
            print(f"   • Tipos de datos:     {dict(esquema)}")
            print("-" * 66)

        except Exception as e:
            print(f"Error al escanear {nombre_csv}: {e}")
            print("-" * 66)
else:
    print(f"Error: La carpeta {carpeta_2022} no existe.")

INSPECCIÓN DE ESQUEMA DE COLUMNAS - MOLINETES 2022
• Analizando archivos en: /content/content/csv/molinetes_2022

Archivo: 202201_PAX15min-ABC.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL': String}
------------------------------------------------------------------
Archivo: 202201_PAX15min-DEH.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL': String}
------------------------------------------------------------------
Archivo: 202202_PAX15min-ABC.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;LINE

In [117]:
# ======================================================================
# CONSOLIDACIÓN DE MOLINETES 2022 (múltiples CSV → un parquet)
# ======================================================================
# Los CSV de 2022 vienen MAL FORMADOS: cada fila entera está envuelta en
# comillas dobles y el separador ';' queda DENTRO de las comillas. Un parser
# normal lo lee como una sola columna gigante. La solución es quote_char=None
# (las comillas pasan a ser texto común) y después limpiarlas a mano.
# Además: pax en minúscula, headers con columnas extra y caracteres rotos.

carpeta_2022 = os.path.abspath(os.path.join("..", "csv", "molinetes_2022"))
ruta_salida_parquet_2022 = os.path.join(CARPETA_PARQUET, "historico_2022.parquet")

print("=" * 70)
print("CONSOLIDACIÓN DE MOLINETES 2022")
print("=" * 70)
print(f"• Carpeta origen:   {carpeta_2022}")
print(f"• Parquet destino:  {ruta_salida_parquet_2022}\n")

if not os.path.exists(carpeta_2022):
    print(f"ERROR: no se encontró la carpeta {carpeta_2022}")
else:
    archivos_csv = sorted(f for f in os.listdir(carpeta_2022) if f.endswith(".csv"))
    lista_frames = []

    print("  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...")
    for nombre_csv in archivos_csv:
        ruta_csv = os.path.join(carpeta_2022, nombre_csv)
        try:
            df_archivo = pl.read_csv(
                ruta_csv,
                separator=";",
                quote_char=None,             # <- CLAVE: ignora las comillas envolventes
                null_values=null_values,
                infer_schema_length=0,       # todo String; se tipa después
                encoding="utf8-lossy",       # tolera caracteres rotos del header
                truncate_ragged_lines=True,
            )

            # Se limpian comillas y espacios de los NOMBRES, y pasamos a MAYÚSCULAS
            # ("FECHA -> FECHA, pax_TOTAL" -> PAX_TOTAL)
            df_archivo = df_archivo.rename(lambda c: c.replace('"', '').strip().upper())

            # Se limpian las comillas que quedaron pegadas en los VALORES de
            # la primera y última columna ("1/1/2022 -> 1/1/2022, 1" -> 1)
            df_archivo = df_archivo.with_columns(
                pl.col(pl.String).str.replace_all('"', '')
            )

            # Seleccionamos solo con las columnas deseadas (descarta HORA/DIA/etc.)
            columnas_a_importar = [c for c in df_archivo.columns if c in COLUMNAS_DESEADAS]

            if "FECHA" not in columnas_a_importar:
                print(f"     [omitido] {nombre_csv}: sin FECHA. Cols: {df_archivo.columns[:5]}")
                continue

            df_archivo = df_archivo.select(columnas_a_importar)
            lista_frames.append(df_archivo)
            print(f"     [ok] {nombre_csv} | filas: {df_archivo.height:,}")

        except Exception as e:
            print(f"     [error] {nombre_csv}: {e}")

    if not lista_frames:
        print("\nERROR: ningún archivo aportó datos válidos.")
    else:
        print(f"\n  [unificación] Concatenando {len(lista_frames)} archivos...")
        df_2022 = pl.concat(lista_frames, how="vertical_relaxed")
        del lista_frames
        gc.collect()
        print(f"  [memoria] Unificado en RAM. Shape inicial: {df_2022.shape}")

        # --- Curación con las funciones del pipeline ---
        df_2022 = estandarizar_columnas(df_2022, 2022)
        df_2022 = tipar_fechas_y_pax(df_2022, 2022)
        df_2022 = tipar_horas(df_2022, 2022)

        # Guardrail: abortar si TODO quedó con FECHA nula (era el bug silencioso)
        nulos_fecha = df_2022["FECHA"].is_null().sum()
        if nulos_fecha == df_2022.height:
            raise ValueError(
                f"ABORTADO: las {df_2022.height:,} filas tienen FECHA nula tras el parseo. "
                "Revisá el formato de fecha antes de exportar."
            )
        print(f"  [control] FECHA nulas tras parseo: {nulos_fecha:,} de {df_2022.height:,}")

        # Consolidación de duplicados por clave (igual que 2020/2021)
        df_2022 = consolidar_duplicados_clave(df_2022, 2022)

        # Diagnósticos
        diagnosticar_duplicados(df_2022, 2022)
        diagnosticar_outliers(df_2022, 2022)

        # Estandarización de LINEA
        df_2022 = estandarizar_linea(df_2022, 2022)

        # Exportación
        df_2022.write_parquet(ruta_salida_parquet_2022, compression="snappy")
        print(f"\n  [export] {ruta_salida_parquet_2022} ({df_2022.height:,} filas)")

        del df_2022
        gc.collect()
        print("  [memoria] RAM del bloque 2022 liberada.")

print("=" * 70)

CONSOLIDACIÓN DE MOLINETES 2022
• Carpeta origen:   /content/content/csv/molinetes_2022
• Parquet destino:  /content/content/parquet/historico_2022.parquet

  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...
     [ok] 202201_PAX15min-ABC.csv | filas: 546,527
     [ok] 202201_PAX15min-DEH.csv | filas: 412,805
     [ok] 202202_PAX15min-ABC.csv | filas: 519,000
     [ok] 202202_PAX15min-DEH.csv | filas: 401,553
     [ok] 202203_PAX15min-ABC.csv | filas: 581,250
     [ok] 202203_PAX15min-DEH.csv | filas: 459,800
     [ok] 202204_PAX15min-ABC.csv | filas: 565,725
     [ok] 202204_PAX15min-DEH.csv | filas: 441,800
     [ok] 202205_PAX15min-ABC.csv | filas: 575,536
     [ok] 202205_PAX15min-DEH.csv | filas: 443,617
     [ok] 202206_PAX15min-ABC.csv | filas: 569,770
     [ok] 202206_PAX15min-DEH.csv | filas: 437,613
     [ok] 202207_PAX15min-ABC.csv | filas: 589,934
     [ok] 202207_PAX15min-DEH.csv | filas: 453,589
     [ok] 202208_PAX15min-ABC.csv | filas: 595,776
  

-------------------------

In [118]:
# ======================================================================
# CONSOLIDACIÓN DE MOLINETES 2023 (múltiples CSV → un parquet)
# ======================================================================
# Los CSV de 2023 vienen MAL FORMADOS: cada fila entera está envuelta en
# comillas dobles y el separador ';' queda DENTRO de las comillas. Un parser
# normal lo lee como una sola columna gigante. La solución es quote_char=None
# (las comillas pasan a ser texto común) y después limpiarlas a mano.
# Además: pax en minúscula, headers con columnas extra y caracteres rotos.

carpeta_2023 = os.path.abspath(os.path.join("..", "csv", "molinetes_2023"))
ruta_salida_parquet_2023 = os.path.join(CARPETA_PARQUET, "historico_2023.parquet")

print("=" * 70)
print("CONSOLIDACIÓN DE MOLINETES 2023")
print("=" * 70)
print(f"• Carpeta origen:   {carpeta_2023}")
print(f"• Parquet destino:  {ruta_salida_parquet_2023}\n")

if not os.path.exists(carpeta_2023):
    print(f"ERROR: no se encontró la carpeta {carpeta_2023}")
else:
    archivos_csv = sorted(f for f in os.listdir(carpeta_2023) if f.endswith(".csv"))
    lista_frames = []

    print("  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...")
    for nombre_csv in archivos_csv:
        ruta_csv = os.path.join(carpeta_2023, nombre_csv)
        try:
            df_archivo = pl.read_csv(
                ruta_csv,
                separator=";",
                quote_char=None,             # <- CLAVE: ignora las comillas envolventes
                null_values=null_values,
                infer_schema_length=0,       # todo String; se tipa después
                encoding="utf8-lossy",       # tolera caracteres rotos del header
                truncate_ragged_lines=True,
            )

            # Se limpian comillas y espacios de los NOMBRES, y pasamos a MAYÚSCULAS
            # ("FECHA -> FECHA, pax_TOTAL" -> PAX_TOTAL)
            df_archivo = df_archivo.rename(lambda c: c.replace('"', '').strip().upper())

            # Se limpian las comillas que quedaron pegadas en los VALORES de
            # la primera y última columna ("1/1/2023 -> 1/1/2023, 1" -> 1)
            df_archivo = df_archivo.with_columns(
                pl.col(pl.String).str.replace_all('"', '')
            )

            # Seleccionamos solo con las columnas deseadas (descarta HORA/DIA/etc.)
            columnas_a_importar = [c for c in df_archivo.columns if c in COLUMNAS_DESEADAS]

            if "FECHA" not in columnas_a_importar:
                print(f"     [omitido] {nombre_csv}: sin FECHA. Cols: {df_archivo.columns[:5]}")
                continue

            df_archivo = df_archivo.select(columnas_a_importar)
            lista_frames.append(df_archivo)
            print(f"     [ok] {nombre_csv} | filas: {df_archivo.height:,}")

        except Exception as e:
            print(f"     [error] {nombre_csv}: {e}")

    if not lista_frames:
        print("\nERROR: ningún archivo aportó datos válidos.")
    else:
        print(f"\n  [unificación] Concatenando {len(lista_frames)} archivos...")
        df_2023 = pl.concat(lista_frames, how="vertical_relaxed")
        del lista_frames
        gc.collect()
        print(f"  [memoria] Unificado en RAM. Shape inicial: {df_2023.shape}")

        # --- Curación con las funciones del pipeline ---
        df_2023 = estandarizar_columnas(df_2023, 2023)
        df_2023 = tipar_fechas_y_pax(df_2023, 2023)
        df_2023 = tipar_horas(df_2023, 2023)

        # Guardrail: abortar si TODO quedó con FECHA nula (era el bug silencioso)
        nulos_fecha = df_2023["FECHA"].is_null().sum()
        if nulos_fecha == df_2023.height:
            raise ValueError(
                f"ABORTADO: las {df_2023.height:,} filas tienen FECHA nula tras el parseo. "
                "Revisá el formato de fecha antes de exportar."
            )
        print(f"  [control] FECHA nulas tras parseo: {nulos_fecha:,} de {df_2023.height:,}")

        # Consolidación de duplicados por clave (igual que 2020/2021)
        df_2023 = consolidar_duplicados_clave(df_2023, 2023)

        # Diagnósticos
        diagnosticar_duplicados(df_2023, 2023)
        diagnosticar_outliers(df_2023, 2023)

        # Estandarización de LINEA
        df_2023 = estandarizar_linea(df_2023, 2023)

        # Exportación
        df_2023.write_parquet(ruta_salida_parquet_2023, compression="snappy")
        print(f"\n  [export] {ruta_salida_parquet_2023} ({df_2023.height:,} filas)")

        del df_2023
        gc.collect()
        print("  [memoria] RAM del bloque 2023 liberada.")

print("=" * 70)

CONSOLIDACIÓN DE MOLINETES 2023
• Carpeta origen:   /content/content/csv/molinetes_2023
• Parquet destino:  /content/content/parquet/historico_2023.parquet

  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...
     [ok] 202301_PAX15min-ABC.csv | filas: 589,514
     [ok] 202301_PAX15min-DEH.csv | filas: 444,453
     [ok] 202302_PAX15min-ABC.csv | filas: 521,836
     [ok] 202302_PAX15min-DEH.csv | filas: 393,305
     [ok] 202303_PAX15min-ABC.csv | filas: 574,225
     [ok] 202303_PAX15min-DEH.csv | filas: 451,465
     [ok] 202304_PAX15min-ABC.csv | filas: 516,058
     [ok] 202304_PAX15min-DEH.csv | filas: 418,182
     [ok] 202305_PAX15min-ABC.csv | filas: 553,490
     [ok] 202305_PAX15min-DEH.csv | filas: 426,590
     [ok] 202306_PAX15min-ABC.csv | filas: 573,435
     [ok] 202306_PAX15min-DEH.csv | filas: 425,707
     [ok] 202307_PAX15min-ABC.csv | filas: 598,368
     [ok] 202307_PAX15min-DEH.csv | filas: 440,460
     [ok] 202308_PAX15min-ABC.csv | filas: 585,223
  

In [119]:
# 1. Se define la ruta de la carpeta que queremos inspeccionar
carpeta_2024 = os.path.abspath(os.path.join("..", "csv", "molinetes_2024"))

print("==================================================================")
print("INSPECCIÓN DE ESQUEMA DE COLUMNAS - MOLINETES 2024")
print("==================================================================")
print(f"• Analizando archivos en: {carpeta_2024}\n")

if os.path.exists(carpeta_2024):
    # Se listan todos los CSVs internos
    archivos_csv = [f for f in os.listdir(carpeta_2024) if f.endswith(".csv")]

    if not archivos_csv:
        print("No se encontraron archivos CSV dentro de la carpeta.")

    for nombre_csv in sorted(archivos_csv):
        ruta_completa = os.path.join(carpeta_2024, nombre_csv)

        try:
            # Se escanean los metadatos de forma Lazy para no tocar la RAM
            lf_inspeccion = pl.scan_csv(ruta_completa, infer_schema_length=5)
            esquema = lf_inspeccion.collect_schema()

            print(f"Archivo: {nombre_csv}")
            print(f"   • Columnas detectadas: {esquema.names()}")
            print(f"   • Tipos de datos:     {dict(esquema)}")
            print("-" * 66)

        except Exception as e:
            print(f"Error al escanear {nombre_csv}: {e}")
            print("-" * 66)
else:
    print(f"Error: La carpeta {carpeta_2024} no existe.")

INSPECCIÓN DE ESQUEMA DE COLUMNAS - MOLINETES 2024
• Analizando archivos en: /content/content/csv/molinetes_2024

Archivo: 202401_PAX15min-ABC.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL': String}
------------------------------------------------------------------
Archivo: 202401_PAX15min-DEH.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL";']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL";': String}
------------------------------------------------------------------
Archivo: 202402_PAX15min-ABC.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;

In [120]:
# ======================================================================
# CONSOLIDACIÓN DE MOLINETES 2024 (múltiples CSV → un parquet)
# ======================================================================
# Los CSV de 2024 vienen MAL FORMADOS: cada fila entera está envuelta en
# comillas dobles y el separador ';' queda DENTRO de las comillas. Un parser
# normal lo lee como una sola columna gigante. La solución es quote_char=None
# (las comillas pasan a ser texto común) y después limpiarlas a mano.
# Además: pax en minúscula, headers con columnas extra y caracteres rotos.

carpeta_2024 = os.path.abspath(os.path.join("..", "csv", "molinetes_2024"))
ruta_salida_parquet_2024 = os.path.join(CARPETA_PARQUET, "historico_2024.parquet")

print("=" * 70)
print("CONSOLIDACIÓN DE MOLINETES 2024")
print("=" * 70)
print(f"• Carpeta origen:   {carpeta_2024}")
print(f"• Parquet destino:  {ruta_salida_parquet_2024}\n")

if not os.path.exists(carpeta_2024):
    print(f"ERROR: no se encontró la carpeta {carpeta_2024}")
else:
    archivos_csv = sorted(f for f in os.listdir(carpeta_2024) if f.endswith(".csv"))
    lista_frames = []

    print("  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...")
    for nombre_csv in archivos_csv:
        ruta_csv = os.path.join(carpeta_2024, nombre_csv)
        try:
            df_archivo = pl.read_csv(
                ruta_csv,
                separator=";",
                quote_char=None,             # <- CLAVE: ignora las comillas envolventes
                null_values=null_values,
                infer_schema_length=0,       # todo String; se tipa después
                encoding="utf8-lossy",       # tolera caracteres rotos del header
                truncate_ragged_lines=True,
            )

            # Se limpian comillas y espacios de los NOMBRES, y pasamos a MAYÚSCULAS
            # ("FECHA -> FECHA, pax_TOTAL" -> PAX_TOTAL)
            df_archivo = df_archivo.rename(lambda c: c.replace('"', '').strip().upper())

            # Se limpian las comillas que quedaron pegadas en los VALORES de
            # la primera y última columna ("1/1/2024 -> 1/1/2024, 1" -> 1)
            df_archivo = df_archivo.with_columns(
                pl.col(pl.String).str.replace_all('"', '')
            )

            # Seleccionamos solo con las columnas deseadas (descarta HORA/DIA/etc.)
            columnas_a_importar = [c for c in df_archivo.columns if c in COLUMNAS_DESEADAS]

            if "FECHA" not in columnas_a_importar:
                print(f"     [omitido] {nombre_csv}: sin FECHA. Cols: {df_archivo.columns[:5]}")
                continue

            df_archivo = df_archivo.select(columnas_a_importar)
            lista_frames.append(df_archivo)
            print(f"     [ok] {nombre_csv} | filas: {df_archivo.height:,}")

        except Exception as e:
            print(f"     [error] {nombre_csv}: {e}")

    if not lista_frames:
        print("\nERROR: ningún archivo aportó datos válidos.")
    else:
        print(f"\n  [unificación] Concatenando {len(lista_frames)} archivos...")
        df_2024 = pl.concat(lista_frames, how="vertical_relaxed")
        del lista_frames
        gc.collect()
        print(f"  [memoria] Unificado en RAM. Shape inicial: {df_2024.shape}")

        # --- Curación con las funciones del pipeline ---
        df_2024 = estandarizar_columnas(df_2024, 2024)
        df_2024 = tipar_fechas_y_pax(df_2024, 2024)
        df_2024 = tipar_horas(df_2024, 2024)

        # Guardrail: abortar si TODO quedó con FECHA nula (era el bug silencioso)
        nulos_fecha = df_2024["FECHA"].is_null().sum()
        if nulos_fecha == df_2024.height:
            raise ValueError(
                f"ABORTADO: las {df_2024.height:,} filas tienen FECHA nula tras el parseo. "
                "Revisá el formato de fecha antes de exportar."
            )
        print(f"  [control] FECHA nulas tras parseo: {nulos_fecha:,} de {df_2024.height:,}")

        # Consolidación de duplicados por clave (igual que 2020/2021)
        df_2024 = consolidar_duplicados_clave(df_2024, 2024)

        # Diagnósticos
        diagnosticar_duplicados(df_2024, 2024)
        diagnosticar_outliers(df_2024, 2024)

        # Estandarización de LINEA
        df_2024 = estandarizar_linea(df_2024, 2024)

        # Exportación
        df_2024.write_parquet(ruta_salida_parquet_2024, compression="snappy")
        print(f"\n  [export] {ruta_salida_parquet_2024} ({df_2024.height:,} filas)")

        del df_2024
        gc.collect()
        print("  [memoria] RAM del bloque 2024 liberada.")

print("=" * 70)

CONSOLIDACIÓN DE MOLINETES 2024
• Carpeta origen:   /content/content/csv/molinetes_2024
• Parquet destino:  /content/content/parquet/historico_2024.parquet

  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...
     [ok] 202401_PAX15min-ABC.csv | filas: 587,990
     [ok] 202401_PAX15min-DEH.csv | filas: 290,788
     [ok] 202402_PAX15min-ABC.csv | filas: 551,054
     [ok] 202402_PAX15min-DEH.csv | filas: 236,273
     [ok] 202403_PAX15min-ABC.csv | filas: 599,344
     [ok] 202403_PAX15min-DEH.csv | filas: 413,562
     [ok] 202404_PAX15min-ABC.csv | filas: 574,669
     [ok] 202404_PAX15min-DEH.csv | filas: 459,968
     [ok] 202405_PAX15min-ABC.csv | filas: 569,524
     [ok] 202405_PAX15min-DEH.csv | filas: 449,116
     [ok] 202406_PAX15min-ABC.csv | filas: 549,191
     [ok] 202406_PAX15min-DEH.csv | filas: 426,072
     [ok] 202407_PAX15min-ABC.csv | filas: 567,561
     [ok] 202407_PAX15min-DEH.csv | filas: 437,985
     [ok] 202408_PAX15min-ABC.csv | filas: 597,421
  

In [121]:
# 1. Se define la ruta de la carpeta que queremos inspeccionar
carpeta_2025 = os.path.abspath(os.path.join("..", "csv", "molinetes_2025"))

print("==================================================================")
print("INSPECCIÓN DE ESQUEMA DE COLUMNAS - MOLINETES 2025")
print("==================================================================")
print(f"• Analizando archivos en: {carpeta_2025}\n")

if os.path.exists(carpeta_2025):
    # Se listan todos los CSVs internos
    archivos_csv = [f for f in os.listdir(carpeta_2025) if f.endswith(".csv")]

    if not archivos_csv:
        print("No se encontraron archivos CSV dentro de la carpeta.")

    for nombre_csv in sorted(archivos_csv):
        ruta_completa = os.path.join(carpeta_2025, nombre_csv)

        try:
            # Se escanean los metadatos de forma Lazy para no tocar la RAM
            lf_inspeccion = pl.scan_csv(ruta_completa, infer_schema_length=5)
            esquema = lf_inspeccion.collect_schema()

            print(f"Archivo: {nombre_csv}")
            print(f"   • Columnas detectadas: {esquema.names()}")
            print(f"   • Tipos de datos:     {dict(esquema)}")
            print("-" * 66)

        except Exception as e:
            print(f"Error al escanear {nombre_csv}: {e}")
            print("-" * 66)
else:
    print(f"Error: La carpeta {carpeta_2025} no existe.")

INSPECCIÓN DE ESQUEMA DE COLUMNAS - MOLINETES 2025
• Analizando archivos en: /content/content/csv/molinetes_2025

Archivo: 202501_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL': String}
------------------------------------------------------------------
Archivo: 202501_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL']
   • Tipos de datos:     {'FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pax_franq;pax_TOTAL': String}
------------------------------------------------------------------
Archivo: 202502_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv
   • Columnas detectadas: ['FECHA;DESDE;HASTA;LINEA;MOLINETE;ESTACION;pax_pagos;pax_pases_pagos;pa

In [122]:
# ======================================================================
# CONSOLIDACIÓN DE MOLINETES 2025 (múltiples CSV → un parquet)
# ======================================================================
# Los CSV de 2025 vienen MAL FORMADOS: cada fila entera está envuelta en
# comillas dobles y el separador ';' queda DENTRO de las comillas. Un parser
# normal lo lee como una sola columna gigante. La solución es quote_char=None
# (las comillas pasan a ser texto común) y después limpiarlas a mano.
# Además: pax en minúscula, headers con columnas extra y caracteres rotos.

carpeta_2025 = os.path.abspath(os.path.join("..", "csv", "molinetes_2025"))
ruta_salida_parquet_2025 = os.path.join(CARPETA_PARQUET, "historico_2025.parquet")

print("=" * 70)
print("CONSOLIDACIÓN DE MOLINETES 2025")
print("=" * 70)
print(f"• Carpeta origen:   {carpeta_2025}")
print(f"• Parquet destino:  {ruta_salida_parquet_2025}\n")

if not os.path.exists(carpeta_2025):
    print(f"ERROR: no se encontró la carpeta {carpeta_2025}")
else:
    archivos_csv = sorted(f for f in os.listdir(carpeta_2025) if f.endswith(".csv"))
    lista_frames = []

    print("  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...")
    for nombre_csv in archivos_csv:

        if "PM" in nombre_csv:
            print(f"     [excluido] {nombre_csv}: corresponde a datos de Premetro.")
            continue

        ruta_csv = os.path.join(carpeta_2025, nombre_csv)
        try:
            df_archivo = pl.read_csv(
                ruta_csv,
                separator=";",
                quote_char=None,             # <- CLAVE: ignora las comillas envolventes
                null_values=null_values,
                infer_schema_length=0,       # todo String; se tipa después
                encoding="utf8-lossy",       # tolera caracteres rotos del header
                truncate_ragged_lines=True,
            )

            # Se limpian comillas y espacios de los NOMBRES, y pasamos a MAYÚSCULAS
            # ("FECHA -> FECHA, pax_TOTAL" -> PAX_TOTAL)
            df_archivo = df_archivo.rename(lambda c: c.replace('"', '').strip().upper())

            # Se limpian las comillas que quedaron pegadas en los VALORES de
            # la primera y última columna ("1/1/2025 -> 1/1/2025, 1" -> 1)
            df_archivo = df_archivo.with_columns(
                pl.col(pl.String).str.replace_all('"', '')
            )

            # Seleccionamos solo con las columnas deseadas (descarta HORA/DIA/etc.)
            columnas_a_importar = [c for c in df_archivo.columns if c in COLUMNAS_DESEADAS]

            if "FECHA" not in columnas_a_importar:
                print(f"     [omitido] {nombre_csv}: sin FECHA. Cols: {df_archivo.columns[:5]}")
                continue

            df_archivo = df_archivo.select(columnas_a_importar)
            lista_frames.append(df_archivo)
            print(f"     [ok] {nombre_csv} | filas: {df_archivo.height:,}")

        except Exception as e:
            print(f"     [error] {nombre_csv}: {e}")

    if not lista_frames:
        print("\nERROR: ningún archivo aportó datos válidos.")
    else:
        print(f"\n  [unificación] Concatenando {len(lista_frames)} archivos...")
        df_2025 = pl.concat(lista_frames, how="vertical_relaxed")
        del lista_frames
        gc.collect()
        print(f"  [memoria] Unificado en RAM. Shape inicial: {df_2025.shape}")

        # --- Curación con las funciones del pipeline ---
        df_2025 = estandarizar_columnas(df_2025, 2025)
        df_2025 = tipar_fechas_y_pax(df_2025, 2025)
        df_2025 = tipar_horas(df_2025, 2025)

        # Guardrail: abortar si TODO quedó con FECHA nula (era el bug silencioso)
        nulos_fecha = df_2025["FECHA"].is_null().sum()
        if nulos_fecha == df_2025.height:
            raise ValueError(
                f"ABORTADO: las {df_2025.height:,} filas tienen FECHA nula tras el parseo. "
                "Revisá el formato de fecha antes de exportar."
            )
        print(f"  [control] FECHA nulas tras parseo: {nulos_fecha:,} de {df_2025.height:,}")

        # Consolidación de duplicados por clave (igual que 2020/2021)
        df_2025 = consolidar_duplicados_clave(df_2025, 2025)

        # Diagnósticos
        diagnosticar_duplicados(df_2025, 2025)
        diagnosticar_outliers(df_2025, 2025)

        # Estandarización de LINEA
        df_2025 = estandarizar_linea(df_2025, 2025)

        # Exportación
        df_2025.write_parquet(ruta_salida_parquet_2025, compression="snappy")
        print(f"\n  [export] {ruta_salida_parquet_2025} ({df_2025.height:,} filas)")

        del df_2025
        gc.collect()
        print("  [memoria] RAM del bloque 2025 liberada.")

print("=" * 70)

CONSOLIDACIÓN DE MOLINETES 2025
• Carpeta origen:   /content/content/csv/molinetes_2025
• Parquet destino:  /content/content/parquet/historico_2025.parquet

  [lectura] Leyendo CSV con quote_char=None (comillas envolventes rotas)...
     [ok] 202501_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv | filas: 517,760
     [ok] 202501_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv | filas: 492,193
     [ok] 202502_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv | filas: 523,055
     [ok] 202502_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv | filas: 403,189
     [ok] 202503_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv | filas: 550,703
     [ok] 202503_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv | filas: 439,467
     [ok] 202504_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv | filas: 536,687
     [ok] 202504_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv | filas: 428,498
     [ok] 202505_PAX15min-ABC-INCLUYEOTROMODOSDEPAGO.csv | filas: 603,608
     [ok] 202505_PAX15min-DEH-INCLUYEOTROMODOSDEPAGO.csv | filas: 475,535
     [ok] 202506_PAX15min-A

In [123]:
'''
# Versión básica de carga de .parquet sin revisar la existencia previa
# --- EJECUCIÓN: un año por vez, sin acumular en RAM ---

procesados = []
for anio in range(2014, 2022):
    if procesar_anio(anio, verbose_nulos=False):
        procesados.append(anio)
    gc.collect()  # seguro adicional entre años

print(f"\n{'='*70}")
print(f"Pipeline finalizado. Años procesados y exportados a Parquet: {procesados}")
print(f"{'='*70}")
'''

'\n# Versión básica de carga de .parquet sin revisar la existencia previa\n# --- EJECUCIÓN: un año por vez, sin acumular en RAM ---\n\nprocesados = []\nfor anio in range(2014, 2022):\n    if procesar_anio(anio, verbose_nulos=False):\n        procesados.append(anio)\n    gc.collect()  # seguro adicional entre años\n\nprint(f"\n{\'=\'*70}")\nprint(f"Pipeline finalizado. Años procesados y exportados a Parquet: {procesados}")\nprint(f"{\'=\'*70}")\n'

-----------------------------------------------------------

### Activación del motor Lazy sobre los Parquet unificados

Una vez exportados todos los años, se mapean de forma perezosa con un único `scan_parquet` y comodín. Esto no carga nada a RAM: arma el plan de ejecución que se materializa recién al hacer `.collect()`.

In [124]:
# Conexión Lazy unificada sobre todos los parquet

ruta_busqueda = os.path.join(CARPETA_PARQUET, "historico_*.parquet")
subte_lazy_df = pl.scan_parquet(ruta_busqueda)

print("Dataset unificado conectado de forma Lazy.")
print(f"• Esquema virtual: {subte_lazy_df.collect_schema().names()}")
print(f"• Tipo de objeto: {type(subte_lazy_df)}")


Dataset unificado conectado de forma Lazy.
• Esquema virtual: ['FECHA', 'DESDE', 'HASTA', 'LINEA', 'ESTACION', 'PAX_TOTAL']
• Tipo de objeto: <class 'polars.lazyframe.frame.LazyFrame'>


## Control de calidad de datos unificados

### Auditoría 1: Volumen y cobertura temporal por año

In [125]:
print("======================================================================")
print("AUDITORÍA 1: VOLUMEN Y COBERTURA TEMPORAL POR AÑO")
print("======================================================================")

Mencion_anios_df = (
    subte_lazy_df
    .with_columns(
        pl.col("FECHA").dt.year().alias("ANIO_EXTRACTO")
    )
    .group_by("ANIO_EXTRACTO")
    .agg([
        pl.len().alias("CANTIDAD_REGISTROS"),
        pl.col("FECHA").null_count().alias("FECHAS_NULAS"),
        pl.col("FECHA").min().alias("FECHA_MINIMA"),
        pl.col("FECHA").max().alias("FECHA_MAXIMA")
    ])
    .sort("ANIO_EXTRACTO")
    .collect()
)

print(Mencion_anios_df)

AUDITORÍA 1: VOLUMEN Y COBERTURA TEMPORAL POR AÑO
shape: (12, 5)
┌───────────────┬────────────────────┬──────────────┬──────────────┬──────────────┐
│ ANIO_EXTRACTO ┆ CANTIDAD_REGISTROS ┆ FECHAS_NULAS ┆ FECHA_MINIMA ┆ FECHA_MAXIMA │
│ ---           ┆ ---                ┆ ---          ┆ ---          ┆ ---          │
│ i32           ┆ u32                ┆ u32          ┆ date         ┆ date         │
╞═══════════════╪════════════════════╪══════════════╪══════════════╪══════════════╡
│ 2014          ┆ 10857244           ┆ 0            ┆ 2014-01-02   ┆ 2014-12-31   │
│ 2015          ┆ 10958582           ┆ 0            ┆ 2015-01-01   ┆ 2015-12-31   │
│ 2016          ┆ 11542322           ┆ 0            ┆ 2016-01-01   ┆ 2016-12-31   │
│ 2017          ┆ 11938476           ┆ 0            ┆ 2017-01-01   ┆ 2017-12-31   │
│ 2018          ┆ 12058191           ┆ 0            ┆ 2018-01-01   ┆ 2018-12-31   │
│ 2019          ┆ 12662343           ┆ 0            ┆ 2019-01-01   ┆ 2019-12-31   │
│ 2020     

### Auditoría 2: Mapeados y unicidad de líneas (A-H)

In [126]:
print("======================================================================")
print("AUDITORÍA 2: MAPEADOS Y UNICIDAD DE LÍNEAS (A-H)")
print("======================================================================")

lineas_auditoria_df = (
    subte_lazy_df
    .group_by("LINEA")
    .agg([
        pl.len().alias("TOTAL_FILAS"),
        pl.col("ESTACION").n_unique().alias("ESTACIONES_DISTINTAS")
    ])
    .sort("LINEA")
    .collect()
)

print(lineas_auditoria_df)

AUDITORÍA 2: MAPEADOS Y UNICIDAD DE LÍNEAS (A-H)
shape: (7, 3)
┌───────┬─────────────┬──────────────────────┐
│ LINEA ┆ TOTAL_FILAS ┆ ESTACIONES_DISTINTAS │
│ ---   ┆ ---         ┆ ---                  │
│ str   ┆ u32         ┆ u32                  │
╞═══════╪═════════════╪══════════════════════╡
│ null  ┆ 1           ┆ 1                    │
│ A     ┆ 17851317    ┆ 44                   │
│ B     ┆ 18805046    ┆ 37                   │
│ C     ┆ 10172449    ┆ 21                   │
│ D     ┆ 17301617    ┆ 43                   │
│ E     ┆ 9427392     ┆ 38                   │
│ H     ┆ 8256470     ┆ 26                   │
└───────┴─────────────┴──────────────────────┘


### Auditoría 3: Integridad de métricas de conteo (pax)

In [127]:
print("======================================================================")
print("AUDITORÍA 3: INTEGRIDAD DE MÉTRICAS DE CONTEO (PAX)")
print("======================================================================")

pax_auditoria_df = (
    subte_lazy_df
    .select([
        # Conteo de nulos por columna
        pl.col("PAX_PAGOS").null_count().alias("PAGOS_NULOS"),
        pl.col("PAX_PASES_PAGOS").null_count().alias("PASES_NULOS"),
        pl.col("PAX_FRANQ").null_count().alias("FRANQ_NULOS"),
        pl.col("PAX_TOTAL").null_count().alias("TOTAL_NULOS"),

        # Conteo de valores negativos aberrantes
        (pl.col("PAX_PAGOS") < 0).sum().alias("PAGOS_NEGATIVOS"),
        (pl.col("PAX_TOTAL") < 0).sum().alias("TOTAL_NEGATIVOS"),

        # Máximos históricos para detectar anomalías de desborde
        pl.col("PAX_TOTAL").max().alias("MAXIMO_PAX_15MIN")
    ])
    .collect()
)

print(pax_auditoria_df)

AUDITORÍA 3: INTEGRIDAD DE MÉTRICAS DE CONTEO (PAX)


ColumnNotFoundError: unable to find column "PAX_PAGOS"; valid columns: ["FECHA", "DESDE", "HASTA", "LINEA", "ESTACION", "PAX_TOTAL"]

Resolved plan until failure:

	---> FAILED HERE RESOLVING 'sink' <---
Parquet SCAN [/content/content/parquet/historico_2014.parquet, ... 11 other sources]
PROJECT */6 COLUMNS
ESTIMATED ROWS: 130286928

### Auditoría 4: Verificación de claves únicas combinadas

In [ ]:
print("======================================================================")
print("AUDITORÍA 4: VERIFICACIÓN DE CLAVES ÚNICAS (MÉTODO PARTICIONADO)")
print("======================================================================")

# Listamos todos los parquets individuales de la carpeta
lista_parquets = sorted(glob.glob(os.path.join(CARPETA_PARQUET, "historico_*.parquet")))

total_claves_duplicadas_serie = 0

print("• Escaneando consistencia de claves año por año...")
print("-" * 70)

for ruta_parquet in lista_parquets:
    nombre_archivo = os.path.basename(ruta_parquet)

    # Leemos el archivo anual de forma ansiosa (Eager) ya que está curado y es liviano
    df_anio = pl.read_parquet(ruta_parquet)

    # Agrupamos por la clave espacio-temporal dentro de este año específico
    duplicados_anio = (
        df_anio
        .group_by(["FECHA", "DESDE", "HASTA", "LINEA", "MOLINETE", "ESTACION"])
        .len()
        .filter(pl.col("len") > 1)
    )

    cantidad_duplicados_anio = duplicados_anio.height
    total_claves_duplicadas_serie += cantidad_duplicados_anio

    status = "OK" if cantidad_duplicados_anio == 0 else f"DETECTADOS: {cantidad_duplicados_anio:,}"
    print(f"{nombre_archivo:<25} | Registros: {df_anio.height:<10,} | Estado: {status}")

    # Liberamos la RAM de este año antes de pasar al siguiente
    del df_anio
    del duplicados_anio
    gc.collect()

print("-" * 70)
print(f"• Auditoría finalizada. Total de claves duplicadas en toda la serie: {total_claves_duplicadas_serie}")

if total_claves_duplicadas_serie == 0:
    print("Ningún año presenta registros duplicados en su clave primaria.")
else:
    print("ATENCIÓN: Se encontraron colisiones que hay que revisar antes de modelar.")
print("======================================================================")

https://github.com/AleLoredo/UGR-metodologia/blob/eze/UGR_Metodologia_TP1_subte.ipynb

In [ ]:
fin_notebook = time.time()
tiempo_total = fin_notebook - inicio_notebook

# Convertir a minutos y segundos para que sea legible
minutos = int(tiempo_total // 60)
segundos = int(tiempo_total % 60)
print(f"Tiempo total de ejecución: {minutos}m {segundos}s (Total: {tiempo_total:.2f} segundos)")